Для этой задачи на Kaggle есть легендарное соревнование. На нем выросло не одно поколение крутых Data Scientist'ов.

- **Название:** **Corporation Favorita Grocery Sales Forecasting** или более простой и популярный аналог: **Store Item Demand Forecasting Challenge**.
- **Что там внутри:** Исторические данные о продажах сотен товаров в разных магазинах по дням.
- **Ваша цель:** Предсказать точное число продаж (Target — непрерывная величина) для каждого товара на 7 дней вперед.
  
В этой задаче 80% успеха — это не выбор модели, а Feature Engineering (создание признаков). 

Мы заставим обычные бустинги (CatBoost или LightGBM) понимать время.

#### Главное правило временных рядов (Важно!)

Забудьте про `train_test_split(X, y, test_size=0.2)`. Если вы перемешаете данные случайно, модель подглядит в будущее (случится **Data Leakage** / утечка данных), покажет идеальный результат на валидации, но полностью провалится на тесте.  
Мы будем валидироваться по принципу **«прошлое учит будущее»**: возьмем первые несколько лет на обучение, а последние 3–6 месяцев — на проверку.

---

#### План

Мы создадим три типа признаков:

1. **Календарные признаки:** День недели (0–6), день месяца (1–31), месяц, является ли день выходным. (Модель быстро поймет, что в субботу таргет должен быть выше).
2. **Лаги (Lag Features):** Добавим колонки `sales_lag_7` (продажи 7 дней назад), `sales_lag_8` и так далее.
3. **Скользящие окна (Rolling Features):** Среднее значение продаж за последние 7 дней, 14 дней, 30 дней. Это покажет модели тренд (продажи растут или падают).

**Метрика качества:** Мы будем оценивать модель через **SMAPE** (Symmetric Mean Absolute Percentage Error) или **MAE** (средняя абсолютная ошибка в штуках товара).

---


Шаг 1. Первичный анализ и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])


sales_by_day = train.groupby(train['date'].dt.day_name())['sales'].sum()
sales_by_items = train.groupby(train['item'])['sales'].sum().sort_values(ascending=False)

SEPARATOR = "=" * 50

print(f"Размер трейна: {train.shape}")
print(SEPARATOR)
print(f"Все продажи по дням недели: {sales_by_day}")
print(SEPARATOR)
print(f"Самые продаваемые товары: {sales_by_items[:5]}")
print(SEPARATOR)
print(f"Обучение с: {train['date'].min()} - {train['date'].max()}")
print(SEPARATOR)
print(f"Тест (что нужно предсказать): с {test['date'].min()} по {test['date'].max()}")
print(SEPARATOR)
print('Уникальные Товары: ', train['item'].unique())
print(SEPARATOR)
print('Уникальные Магазины: ', train['store'].unique())
print(SEPARATOR)
print('Количество продаж всего: ', train['sales'].sum())
print(SEPARATOR)
print('Самый продоваемая запись: ', train.nlargest(1, 'sales'))
print(SEPARATOR)
print('Самый непродоваемая запись: ', train.nsmallest(1, 'sales'))
print(SEPARATOR)


Шаг 2. Feature Engineering: Календарные признаки

In [ ]:
def create_calendar_features(df):
    df['sub_date'] = df['date'] # сохраняем копию
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int) # день является началом месяца?
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int) # день является концом месяца?
    # df['is_month_middle'] = (df['day'] == 15).astype(int)
    return df

train = create_calendar_features(train)
test = create_calendar_features(test)

print(train[['date', 'dayofweek', 'is_weekend', 'is_month_start', 'is_month_end',]].head())

Шаг 3. Генерация лагов (Lag Features) и скользящих средних (Rolling Mean)

In [ ]:
# Обьединяем датасеты, у тест-набора продаж пока нет (будут NaN)
df = pd.concat([train, test], sort=False).reset_index(drop=True)

# Сортируем по магазину, товару и дате, чтобы сдвиги во времени шли правильно
df.sort_values(by=['store', 'item', 'date'], inplace=True)

# 1. Создаем Lag Features (временные сдвиги)
# Смотрим на продажи 91 день назад, 98 дней назад и 365 дней назад для каждой записи
# 91 - краткосрочный тренд, 98 - недельную цикличность, 365 - глобальный масштаб товара
for lag in [91, 98, 365]:
    df[f'sales_lag_{lag}'] = df.groupby(['store', 'item'])['sales'].shift(lag)

print(df.tail())

# 2. Создаем Rolling Features (скользящее среднее)
# Чтобы не было утечки данных (Data Leakage), берем окно от безопасного лага (91 день) - горизонт прогноза
# Среднее арифметическое продаж за 3 месяца назад (окно)
for window in [91, 98, 365]:
    df[f'sales_roll_mean_{window}'] = df.groupby(['store', 'item'])['sales'].shift(91).transform(lambda x: x.rolling(window).mean())

print(df.columns)


Шаг 4. Валидация «Прошлое против Будущего» (Time-Based Split)

Теперь разделяем наш большой датасет обратно на `train` и `test`,  а сам `train` делим на обучающую выборку и валидационную (на которой будем проверять качество регрессии).

Поскольку тест длится 3 месяца, для честной валидации мы отрежем от конца `train`  тоже **последние 3 месяца**.

In [ ]:
# Разделяем обратно на train и test
train_df = df[df['sales'].notnull()]
test_df = df[df['sales'].isnull()]

# Находим границу для валидации (отнимаем 3 месяца от max даты трейна)
max_date = train_df['date'].max()
val_start_date = max_date - pd.Timedelta(days=90)

# Делим train на обучающую выборку и валидационную
X_train = train_df[train_df['date'] < val_start_date]
X_val = train_df[train_df['date'] >= val_start_date]

# Дропаем колонки, которые модели нельзя давать (id, date, target)
features = [col for col in train_df.columns if col not in ['id', 'date', 'sales', 'sub_date']]

y_train = X_train['sales']
y_val = X_val['sales']

X_train = X_train[features]
X_val = X_val[features]

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер валидационной выборки: {X_val.shape}")


Шаг 5. Обучение LightGBM (Регрессия)

Теперь запускаем **LightGBM** в режиме **Регрессии**. Наша задача — предсказать точное количество проданных товаров (`sales`).

В качестве метрики ошибки мы возьмем **MAE (Mean Absolute Error)**. Она показывает среднюю ошибку в реальных единицах (штуках товара). Если MAE равно 5, значит, модель в среднем ошибается на 5 товаров в день.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# 1. Создаем специальные структуры данных для LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# 2. Задаем параметры для задачи Регрессии
params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'random_state': 42,
    'n_jobs': -1,
}

# 3. Обучаем модель с ранней остановкой (Early Stopping)
model_lgb = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=50)]
)

# 4. Делаем предсказание на валидационной выборке
val_preds = model_lgb.predict(X_val)

# 5. Считаем ошибку МАЕ
mae = mean_absolute_error(y_val, val_preds)
print(f"Средняя ошибка (МАЕ) на валидации: {mae:.4f} шт. товара")

Training until validation scores don't improve for 50 rounds
[50]	training's l1: 7.41907	valid_1's l1: 7.19831
[100]	training's l1: 6.5948	valid_1's l1: 6.52526
[150]	training's l1: 6.32311	valid_1's l1: 6.3076
[200]	training's l1: 6.1732	valid_1's l1: 6.18487
[250]	training's l1: 6.06924	valid_1's l1: 6.10069
[300]	training's l1: 5.99037	valid_1's l1: 6.04162
[350]	training's l1: 5.93211	valid_1's l1: 5.99425
[400]	training's l1: 5.8803	valid_1's l1: 5.96058
[450]	training's l1: 5.84464	valid_1's l1: 5.93337
[500]	training's l1: 5.81397	valid_1's l1: 5.91472
[550]	training's l1: 5.78964	valid_1's l1: 5.90014
[600]	training's l1: 5.76962	valid_1's l1: 5.88666
[650]	training's l1: 5.75203	valid_1's l1: 5.87514
[700]	training's l1: 5.73555	valid_1's l1: 5.86662
[750]	training's l1: 5.72262	valid_1's l1: 5.86083
[800]	training's l1: 5.71103	valid_1's l1: 5.85612
[850]	training's l1: 5.70152	valid_1's l1: 5.85175
[900]	training's l1: 5.6917	valid_1's l1: 5.84805
[950]	training's l1: 5.6833

Шаг 6. Анализ важности признаков (Feature Importance)

Одно из главных преимуществ бустингов перед нейросетями — мы можем заглянуть   
внутрь и понять  какие признаки оказались самыми полезными для прогноза продаж.

In [12]:
import matplotlib.pyplot as plt

# Получаем важность признаков
importance = pd.DataFrame({
    'feature': features,
    'importance': model_lgb.feature_importance(importance_type='gain')
}).sort_values(by='importance', ascending=False)

print("\nТоп-10 самых важных признаков для прогноза:")
print(importance.head(10))


Топ-10 самых важных признаков для прогноза:
                feature    importance
11        sales_lag_365  3.344530e+09
9          sales_lag_91  1.580750e+09
10         sales_lag_98  6.802122e+08
3                 month  3.833186e+08
5             dayofweek  3.691743e+08
14  sales_roll_mean_365  3.038716e+08
1                  item  8.323712e+07
13   sales_roll_mean_98  6.253909e+07
12   sales_roll_mean_91  5.181823e+07
2                  year  2.306252e+07


Шаг 7. Сравнение с XGBoost (Регрессия)
- строит деревья по уровням **level-wise** а не вглубь по листьям **leaf-wise**
- имеет встроенную L1/L2 регуляризацию

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import time

print("Старт обучения XGBoost...")
start_time = time.time()

# 1. Инициализируем модель XGBoost в режиме регрессии
# Добавляем сильные регуляризаторы (reg_alpha, reg_lambda)
model_xgb = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0, # l1 регуляризация (аналог Lasso)
    reg_lambda=3.0, # L2 регуляризация (аналог Ridge)
    random_state=42,
    n_jobs=-1,
    eval_metric='mae',
    early_stopping_rounds=50,
)

model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=100
)

xgb_time = time.time() - start_time
print(f"XGBoost обучился за {xgb_time:.2f} секунд.")

xgb_preds = model_xgb.predict(X_val)
xgb_mae = mean_absolute_error(y_val, xgb_preds)

# xgboost победил за счет регуляризации
print(f"Средняя ошибка (MAE) на валидации: {xgb_mae:.4f} шт. товара")
print(f"Для справки, MAE LightGBM было: {mae:.4f}")

Старт обучения XGBoost...
[0]	validation_0-mae:22.20185	validation_1-mae:21.91590
[100]	validation_0-mae:6.77904	validation_1-mae:6.46710
[200]	validation_0-mae:6.44839	validation_1-mae:6.17656
[300]	validation_0-mae:6.28617	validation_1-mae:6.03535
[400]	validation_0-mae:6.17072	validation_1-mae:5.95468
[500]	validation_0-mae:6.07256	validation_1-mae:5.90343
[600]	validation_0-mae:6.00641	validation_1-mae:5.87226
[700]	validation_0-mae:5.93537	validation_1-mae:5.85192
[800]	validation_0-mae:5.87946	validation_1-mae:5.83863
[900]	validation_0-mae:5.83840	validation_1-mae:5.82898
[999]	validation_0-mae:5.80115	validation_1-mae:5.82422
XGBoost обучился за 18.16 секунд.
Средняя ошибка (MAE) на валидации: 5.8241 шт. товара
Для справки, MAE LightGBM было: 5.8417


#### Блендинг (Ансамблирование)

В начале пути я пробовал сложный **стэкинг** (обучение мета-модели поверх предсказаний).  
Но на практике и в соревнованиях Kaggle часто используют более простой, быстрый и  
эффективный приём — **Блендинг (Blending)**.

Это простое взвешенное среднее предсказаний разных моделей. Так как LightGBM и XGBoost  
по-разному строят деревья (одно вглубь по листьям, другое по уровням), они ошибаются в  
разных местах. Объединив их, мы заставим их ошибки «взаимно уничтожиться».

Попробуем смешать их в пропорции **60% XGBoost (так как он точнее) и 40% LightGBM**:

In [17]:
# 1. Объединяем предсказания с весами
blended_preds = 0.60 * xgb_preds + 0.40 * val_preds

# 2. Считаем новую ошибку
blended_mae = mean_absolute_error(y_val, blended_preds)

print(f"MAE LightGBM: {mae:.4f}")
print(f"MAE XGBoost:  {xgb_mae:.4f}")
print(f"--------------------------------")
print(f" Финальный MAE Блендинга: {blended_mae:.4f}")

# Проверим, стал ли результат лучше лучшей модели
if blended_mae < min(mae, xgb_mae):
    print("\n Успех! Ансамбль побил обе модели и дал лучшую точность!")
else:
    print("\n Смешивание не дало прироста. Нужно подбирать другие веса.")

MAE LightGBM: 5.8417
MAE XGBoost:  5.8241
--------------------------------
 Финальный MAE Блендинга: 5.8189

 Успех! Ансамбль побил обе модели и дал лучшую точность!
